In [80]:
import requests
import pandas as pd
import time
import os
from datetime import datetime

In [81]:
weather_api="https://archive-api.open-meteo.com/v1/archive"
AQI_api="https://air-quality-api.open-meteo.com/v1/air-quality"

In [82]:
Output_path="data/weather_aqi_data.csv"

In [83]:
start_date="2015-01-01"
end_date="2026-01-01"

In [84]:
Latitude=18.52
Longitude=73.85

In [85]:
Weather_Vars=[
    "temperature_2m",
    "relative_humidity_2m",
    "wind_speed_10m",
    "wind_direction_10m",
    "precipitation",
]

In [86]:
AQI_VARS = [
    "pm2_5",
    "pm10",
    "nitrogen_dioxide",
    "ozone",
    "carbon_monoxide",
    "european_aqi",
]

In [87]:
def featch_weather(lat,long,start,end):
    print(f"Fetching data of ({start} -> {end})")

    params={
        "latitude":lat,
        "longitude":long,
        "start_date":start,
        "end_date":end,
        "hourly": ",".join(Weather_Vars),
        "timezone":"Asia/Kolkata",
        "wind_speed_unit":"kmh",
    }

    resp=requests.get(weather_api,params=params,timeout=60)
    resp.raise_for_status()
    data=resp.json()
    df=pd.DataFrame(data["hourly"])
    df.rename(columns={
        "time":                   "timestamp",
        "temperature_2m":         "temperature",
        "relative_humidity_2m":   "humidity",
        "wind_speed_10m":         "wind_speed",
        "wind_direction_10m":     "wind_direction",
        "precipitation":          "precipitation",
    }, inplace=True)
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    print(f" Weather rows fetched: {len(df):,}")
    return df

In [88]:
def fetch_aqi(lat,long,start,end):
    aqi_frame=[]
    years=list(range(int(start[:4]),int(end[:4])+1))
    
    for year in years:
        for x in range(1,13,3):
            a=31
            if(x+2==6 or x+2==9 or x+2==4 or x+2==11):
                a=30
            y_start=f"{year}-{x:02d}-01"
            y_end=f"{year}-{x+2:02d}-{a}"
            y_start=max(y_start,start)
            y_end=min(y_end,end)
            if y_start > y_end:
                continue
            params={
                "longitude":long,
                "latitude":lat,
                "start_date":y_start,
                "end_date":y_end,
                "hourly":",".join(AQI_VARS),
                "timezone":"Asia/Kolkata",
            }
            resp=requests.get(AQI_api,params=params,timeout=120)
            resp.raise_for_status()
            data=resp.json()
            if "hourly" not in data:
                print("No data for:", y_start, y_end)
                continue
            df_year=pd.DataFrame(data["hourly"])
            aqi_frame.append(df_year)
            time.sleep(0.1)
    if not aqi_frame:
        return pd.DataFrame()
    df=pd.concat(aqi_frame,ignore_index=True)
    df.rename(columns={
        "time":               "timestamp",
        "nitrogen_dioxide":   "no2",
        "european_aqi":       "aqi",
    }, inplace=True)
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    print(f"Total AQI rows fetched: {len(df):,}")
    return df

In [89]:
def merge_and_clean(df_weather,df_aqi):
    print("Merging and Cleaning Data\n")

    df=pd.merge(df_weather,df_aqi,how="inner",on="timestamp")

    before=len(df)

    df.set_index("timestamp",inplace=True)
    df.sort_index(inplace=True)

    df.ffill(limit=3,inplace=True)
    df.bfill(limit=3,inplace=True)

    df.dropna(inplace=True)

    after=len(df)

    print(f" Rows after merge: {before:,}  →  after cleaning: {after:,}")

    df["aqi_category"]=df["aqi"].apply(aqi_category)

    df.reset_index(inplace=True)
    return df

In [90]:
def aqi_category(value):
    if value <= 20:   return "Good"
    elif value <= 40: return "Fair"
    elif value <= 60: return "Moderate"
    elif value <= 80: return "Poor"
    elif value <= 100:return "Very Poor"
    else:             return "Extremely Poor"
 
 

In [91]:
def save(df, path):
    print(f"Saving to {path} ...")
    os.makedirs(os.path.dirname(path), exist_ok=True)
    df.to_csv(path, index=False)
    print(f"    ✓ Saved {len(df):,} rows × {len(df.columns)} columns")

In [93]:
if __name__ == "__main__":
    print("\n🌦️  Weather + AQI Data Fetcher")
    print(f"   Location : ({Latitude}, {Longitude})")
    print(f"   Period   : {start_date}  →  {end_date}\n")
 
    df_weather = featch_weather(Latitude, Longitude, start_date,end_date)
    df_aqi     = fetch_aqi(Latitude, Longitude, start_date,end_date)
    df         = merge_and_clean(df_weather, df_aqi)
 
    save(df, Output_path)
 
    print("\n✅ Done! Your dataset is ready at:", Output_path)


🌦️  Weather + AQI Data Fetcher
   Location : (18.52, 73.85)
   Period   : 2015-01-01  →  2026-01-01

Fetching data of (2015-01-01 -> 2026-01-01)
 Weather rows fetched: 96,456
Total AQI rows fetched: 96,456
Merging and Cleaning Data

 Rows after merge: 96,456  →  after cleaning: 29,902
Saving to data/weather_aqi_data.csv ...
    ✓ Saved 29,902 rows × 13 columns

✅ Done! Your dataset is ready at: data/weather_aqi_data.csv
